# FIVB Rules PDF Chunking with Docling

Prepare chunks from the cleaned FIVB volleyball rules PDF for later Ollama embedding and vector store ingestion.

In [1]:
from pathlib import Path
import json
from pprint import pprint

from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

PDF_PATH = Path("data/(Cleaned) FIVB-Volleyball_Rules2025_2028-EN-v05-9-87.pdf")
TOKENIZER_MODEL = "Qwen/Qwen3-Embedding-0.6B"
MAX_TOKENS = 900
OUT_PATH = Path("cleaned_fivb_rules_chunks.jsonl")

In [2]:
assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"

size_mb = PDF_PATH.stat().st_size / (1024 * 1024)
print(f"PDF: {PDF_PATH}")
print(f"Size: {size_mb:.2f} MB")

PDF: data/(Cleaned) FIVB-Volleyball_Rules2025_2028-EN-v05-9-87.pdf
Size: 4.56 MB


In [3]:
converter = DocumentConverter()

tokenizer = HuggingFaceTokenizer.from_pretrained(
    model_name=TOKENIZER_MODEL,
    max_tokens=MAX_TOKENS,
)

chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True,
)

In [4]:
def make_record(chunk, chunk_index):
    metadata = chunk.meta.export_json_dict()
    return {
        "id": f"cleaned_fivb_rules:{chunk_index}",
        "source": str(PDF_PATH),
        "chunk_index": chunk_index,
        "text": chunker.contextualize(chunk),
        "raw_text": chunk.text,
        "metadata": metadata,
    }

In [5]:
# Dry run: convert the PDF, create chunks, and inspect one chunk before saving anything.
doc = converter.convert(PDF_PATH).document
chunks = list(chunker.chunk(dl_doc=doc))

assert chunks, "No chunks were generated"

first_record = make_record(chunks[0], 0)

print(f"Generated {len(chunks)} chunks")
print("\nRaw text preview:\n")
print(first_record["raw_text"][:1200])
print("\nContextualized text preview:\n")
print(first_record["text"][:1200])
print("\nMetadata preview:\n")
pprint({
    "id": first_record["id"],
    "source": first_record["source"],
    "headings": first_record["metadata"].get("headings"),
    "captions": first_record["metadata"].get("captions"),
    "doc_item_count": len(first_record["metadata"].get("doc_items", [])),
})

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Generated 136 chunks

Raw text preview:

Volleyball is a sport played by two teams on a playing court divided by a net. There are different versions available for specific circumstances in order to offer the versatility of the game to everyone.
The object of the game is to send the ball over the net in order to ground it on the opponent's court, and to prevent the same effort by the opponent. The team has three hits for returning the ball (in addition to the block contact).
The ball is put in play with a service: hit by the server over the net to the opponents. The rally continues until the ball is grounded on the playing court, goes 'out' or a team fails to return it properly.
In  Volleyball,  the  team  winning  a  rally  scores  a  point  (Rally  Point  System). When the receiving team wins a rally, it gains a point and the right to serve, and its players rotate one position clockwise.
AVE
+ + +
+
+ + +

Contextualized text preview:

GAME CHARACTERISTICS
Volleyball is a sport played

In [6]:
records = [make_record(chunk, index) for index, chunk in enumerate(chunks)]

print(f"Prepared {len(records)} JSONL records")

Prepared 136 JSONL records


In [7]:
with OUT_PATH.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(records)} chunks to {OUT_PATH}")

Saved 136 chunks to cleaned_fivb_rules_chunks.jsonl


In [8]:
text_lengths = [len(record["text"]) for record in records]

print(f"Chunk count: {len(records)}")
print(f"Min chars: {min(text_lengths)}")
print(f"Max chars: {max(text_lengths)}")
print(f"Avg chars: {sum(text_lengths) / len(text_lengths):.0f}")

print("\nFirst 3 chunk headings:\n")
for record in records[:3]:
    pprint({
        "id": record["id"],
        "headings": record["metadata"].get("headings"),
    })

Chunk count: 136
Min chars: 32
Max chars: 3176
Avg chars: 944

First 3 chunk headings:

{'headings': ['GAME CHARACTERISTICS'], 'id': 'cleaned_fivb_rules:0'}
{'headings': ['PART 1 PHILOSOPHY OF RULES AND REFEREEING'],
 'id': 'cleaned_fivb_rules:1'}
{'headings': ['INTRODUCTION'], 'id': 'cleaned_fivb_rules:2'}
